In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess

if not os.path.exists('/content/hamer'):
    subprocess.run('git clone --recursive https://github.com/geopavlakos/hamer.git',
                   shell=True, cwd='/content')

%cd /content/hamer

if not os.path.exists('hamer.egg-info'):
    !pip install -e .[all]
    !pip install -v -e third-party/ViTPose

if not os.path.exists('_DATA/data'):
    !bash fetch_demo_data.sh

os.makedirs('_DATA/data/mano', exist_ok=True)
if not os.path.exists('_DATA/data/mano/MANO_RIGHT.pkl'):
    !cp /content/drive/MyDrive/hamer_assets/mano/MANO_RIGHT.pkl _DATA/data/mano/

# Separating example_data from my_data
os.makedirs('my_data', exist_ok=True)
!cp -n /content/drive/MyDrive/hamer_assets/samples/*.* my_data/ 2>/dev/null

print("completed.")

print('--- official (5 pic) ---')
!ls example_data/

print('--- user (7 pic) ---')
!ls my_data/

Mounted at /content/drive
/content/hamer
Obtaining file:///content/hamer
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/facebookresearch/detectron2 to /tmp/pip-install-d47z87to/detectron2_bb517a0d53944781a6de4a9b81e787cb
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/detectron2 /tmp/pip-install-d47z87to/detectron2_bb517a0d53944781a6de4a9b81e787cb
  Resolved https://github.com/facebookresearch/detectron2 to commit b4a4a3bd136852dae5fb1de37978dee412653e31
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/mattloper/chumpy to /tmp/pip-install-d47z87to/chumpy_4e3c1a10abf04c57b056f416a5643f1f
  Running command git clone --filter=blob:none --quiet https://github.com/mattloper/chumpy /tmp/pip-install-d47z87to/chumpy_4e3c1a10abf04c57b056f416a5643f1f
  Resolved https://github.com/mattloper/chumpy to commit 580566eafc9ac68b2614b64d6f7aaa84eebb70da
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━

In [ ]:
!python demo.py --img_folder example_data --out_folder out_official \
    --side_view --save_mesh --full_frame

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
apex is not installed
apex is not installed
apex is not installed
/usr/local/lib/python3.12/dist-packages/mmcv/cnn/bricks/transformer.py:27: UserWarning: Fail to import ``MultiScaleDeformableAttention`` from ``mmcv.ops.multi_scale_deform_attn``, You should install ``mmcv-full`` if you need this module. 
  warnings.warn('Fail to import ``MultiScaleDeformableAttention`` from '
  [============================================================] 100.0% of 5757.9MB file  
Extracting file: hamer_demo_data.tar.gz
_DATA/
_DATA/vitpose_ckpts/
_DATA/vitpose_ckpts/vitpose+_huge/
_DATA/vitpose_ckpts/vitpose+_huge/wholebody.pth
_DATA/data/
_DATA/data/mano_mean_params.npz
_DATA/data/mano/
_DATA/hamer_ckpts/
_DATA/hamer_ckpts/model_c

In [ ]:
src = open('demo.py').read()

# [A] person detection count
a_old = """        # Detect human keypoints for each person
        vitposes_out = cpm.predict_pose("""
a_new = """        print(f'[DBG] === {os.path.basename(str(img_path))} ===')
        print(f'[DBG]   persons={len(pred_bboxes)} scores={np.round(pred_scores, 3).tolist()}')

        # Detect human keypoints for each person
        vitposes_out = cpm.predict_pose("""

# [B] hand keypoint statistics + candidate IoU
b_old = """        for vitposes in vitposes_out:
            left_hand_keyp = vitposes['keypoints'][-42:-21]
            right_hand_keyp = vitposes['keypoints'][-21:]

            # Rejecting not confident detections"""
b_new = """        for vitposes in vitposes_out:
            left_hand_keyp = vitposes['keypoints'][-42:-21]
            right_hand_keyp = vitposes['keypoints'][-21:]

            _cand = []
            for _s, _kp in (('L', left_hand_keyp), ('R', right_hand_keyp)):
                _v = _kp[:, 2] > 0.5
                _n = int(sum(_v))
                _mc = float(_kp[_v, 2].mean()) if _n > 0 else 0.0
                print(f'[DBG]   {_s}: valid_kp={_n}/21 (need>3), '
                      f'mean_conf={_mc:.3f}, max_conf={float(_kp[:, 2].max()):.3f}')
                if _n > 3:
                    _cand.append([_kp[_v, 0].min(), _kp[_v, 1].min(),
                                  _kp[_v, 0].max(), _kp[_v, 1].max()])

            if len(_cand) == 2:
                _b1, _b2 = _cand
                _iw = max(0, min(_b1[2], _b2[2]) - max(_b1[0], _b2[0]))
                _ih = max(0, min(_b1[3], _b2[3]) - max(_b1[1], _b2[1]))
                _it = _iw * _ih
                _u = ((_b1[2]-_b1[0])*(_b1[3]-_b1[1])
                      + (_b2[2]-_b2[0])*(_b2[3]-_b2[1]) - _it)
                print(f'[DBG]   >> L/R candidate IoU = {(_it/_u if _u > 0 else 0):.3f}')

            # Rejecting not confident detections"""

# [C] bbox count which is handed to HaMeR
c_old = """        if len(bboxes) == 0:
            continue"""
c_new = """        if len(bboxes) == 0:
            print('[DBG]   -> NO HAND BBOX: HaMeR not executed')
            continue
        print(f'[DBG]   -> {len(bboxes)} bbox(es) to HaMeR, is_right={is_right}')"""

out, ok = src, True
for n, o, w in [('A', a_old, a_new), ('B', b_old, b_new), ('C', c_old, c_new)]:
    if o in out:
        out = out.replace(o, w, 1); print(f'patch {n}: OK')
    else:
        ok = False; print(f'patch {n}: FAILED')

if ok:
    open('demo_debug.py', 'w').write(out)
    print('\nSUCCESS: demo_debug.py created')

patch A: OK
patch B: OK
patch C: OK

SUCCESS: demo_debug.py created


In [ ]:
!python -u demo_debug.py --img_folder my_data --out_folder out_debug \
    --side_view --save_mesh --full_frame 2>&1 | tee debug_log.txt

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
apex is not installed
apex is not installed
apex is not installed
/usr/local/lib/python3.12/dist-packages/mmcv/cnn/bricks/transformer.py:27: UserWarning: Fail to import ``MultiScaleDeformableAttention`` from ``mmcv.ops.multi_scale_deform_attn``, You should install ``mmcv-full`` if you need this module. 
  warnings.warn('Fail to import ``MultiScaleDeformableAttention`` from '
Use load_from_local loader
The model and loaded state dict do not match exactly

unexpected key in source state_dict: backbone.blocks.0.mlp.experts.0.weight, backbone.blocks.0.mlp.experts.0.bias, backbone.blocks.0.mlp.experts.1.weight, backbone.blocks.0.mlp.experts.1.bias, backbone.blocks.0.mlp.experts.2.weight, backbone.blocks.0.mlp.experts.2.b

In [ ]:
import re, pandas as pd

rows, cur = [], None
for line in open('debug_log.txt'):
    line = line.strip()
    if m := re.match(r'\[DBG\] === (.+) ===', line):
        if cur: rows.append(cur)
        cur = {'image': m.group(1), 'persons': 0, 'L_kp': 0, 'R_kp': 0,
               'L_mean': 0.0, 'R_mean': 0.0, 'L_max': 0.0, 'R_max': 0.0,
               'IoU': None, 'to_hamer': 0}
        continue
    if cur is None:
        continue
    if m := re.search(r'persons=(\d+)', line):
        cur['persons'] = int(m.group(1))
    elif m := re.match(r'\[DBG\]\s+([LR]): valid_kp=(\d+)/21.*'
                       r'mean_conf=([\d.]+), max_conf=([\d.]+)', line):
        s = m.group(1)
        cur[f'{s}_kp']   = int(m.group(2))
        cur[f'{s}_mean'] = float(m.group(3))
        cur[f'{s}_max']  = float(m.group(4))
    elif m := re.search(r'candidate IoU = ([\d.]+)', line):
        cur['IoU'] = float(m.group(1))
    elif m := re.search(r'-> (\d+) bbox\(es\)', line):
        cur['to_hamer'] = int(m.group(1))
if cur: rows.append(cur)

def diagnose(r):
    if r['to_hamer'] >= 2: return 'DUPLICATE: both L/R candidates passed'
    if r['to_hamer'] == 1: return 'OK'
    if r['persons'] == 0:  return 'STAGE-1 failed: person detection 0'
    return 'STAGE-2 failed: insufficient hand keypoint confidence'

df = pd.DataFrame(rows)
df['diagnosis'] = df.apply(diagnose, axis=1)
df.to_csv('results_diagnosis.csv', index=False)
display(df)

,image,persons,L_kp,R_kp,L_mean,R_mean,L_max,R_max,IoU,to_hamer,diagnosis
0,fist_closeup_1.jpg,1,3,0,0.529,0.000,0.578,0.485,NaN,0,STAGE-2 failed: insufficient hand keypoint con...
1,fist_closeup_2.jpg,1,0,0,0.000,0.000,0.460,0.390,NaN,0,STAGE-2 failed: insufficient hand keypoint con...
2,palm_closeup_1.jpg,1,7,5,0.556,0.561,0.590,0.616,0.696,2,DUPLICATE: both L/R candidates passed
3,normal_hand_2.jpg,1,21,20,0.796,0.641,0.975,0.795,0.889,2,DUPLICATE: both L/R candidates passed
4,normal_hand_1.jpg,1,21,11,0.703,0.610,0.952,0.848,0.795,2,DUPLICATE: both L/R candidates passed
5,two_hands_overlap.jpg,1,2,0,0.563,0.000,0.587,0.496,NaN,0,STAGE-2 failed: insufficient hand keypoint con...
6,ambiguous_hand_1.jpg,1,2,2,0.603,0.563,0.643,0.590,NaN,0,STAGE-2 failed: insufficient hand keypoint con...


In [ ]:
src = open('demo.py').read()

original_block = """        for vitposes in vitposes_out:
            left_hand_keyp = vitposes['keypoints'][-42:-21]
            right_hand_keyp = vitposes['keypoints'][-21:]

            # Rejecting not confident detections
            keyp = left_hand_keyp
            valid = keyp[:,2] > 0.5
            if sum(valid) > 3:
                bbox = [keyp[valid,0].min(), keyp[valid,1].min(), keyp[valid,0].max(), keyp[valid,1].max()]
                bboxes.append(bbox)
                is_right.append(0)
            keyp = right_hand_keyp
            valid = keyp[:,2] > 0.5
            if sum(valid) > 3:
                bbox = [keyp[valid,0].min(), keyp[valid,1].min(), keyp[valid,0].max(), keyp[valid,1].max()]
                bboxes.append(bbox)
                is_right.append(1)"""

TEMPLATE = """        for vitposes in vitposes_out:
            left_hand_keyp = vitposes['keypoints'][-42:-21]
            right_hand_keyp = vitposes['keypoints'][-21:]

            # [MOD] Collect candidates instead of committing them immediately
            candidates = []
            for side_flag, keyp in ((0, left_hand_keyp), (1, right_hand_keyp)):
                valid = keyp[:,2] > 0.5
                if sum(valid) > 3:
                    bbox = [keyp[valid,0].min(), keyp[valid,1].min(),
                            keyp[valid,0].max(), keyp[valid,1].max()]
                    candidates.append({'bbox': bbox,
                                       'is_right': side_flag,
                                       'n_valid': int(sum(valid)),
                                       'conf': float(keyp[valid,2].mean())})

            # [MOD] Remove duplication based on IoU (greedy NMS)
            def _iou(b1, b2):
                iw = max(0, min(b1[2], b2[2]) - max(b1[0], b2[0]))
                ih = max(0, min(b1[3], b2[3]) - max(b1[1], b2[1]))
                inter = iw * ih
                union = ((b1[2]-b1[0])*(b1[3]-b1[1])
                         + (b2[2]-b2[0])*(b2[3]-b2[1]) - inter)
                return inter / union if union > 0 else 0.0

            filtered = []
            for c in sorted(candidates, key=RANK_KEY, reverse=True):
                if all(_iou(c['bbox'], k['bbox']) < IOU_THRESH for k in filtered):
                    filtered.append(c)

            if len(candidates) > len(filtered):
                print(f'[NMS] {os.path.basename(str(img_path))} [VERSION]: '
                      f'{len(candidates)} -> {len(filtered)} | '
                      f'kept is_right={[c["is_right"] for c in filtered]} | '
                      f'cand=' + str([(c['is_right'], c['n_valid'], round(c['conf'], 3))
                                      for c in candidates]))

            for c in filtered:
                bboxes.append(c['bbox'])
                is_right.append(c['is_right'])"""

VERSIONS = {
    'demo_nms.py':    ("lambda c: c['conf']",                 'v1-conf'),
    'demo_nms_v2.py': ("lambda c: (c['n_valid'], c['conf'])", 'v2-count'),
}

if original_block not in src:
    print('FAILED: anchor not found')
else:
    for fn, (rank, tag) in VERSIONS.items():
        blk = TEMPLATE.replace('RANK_KEY', rank).replace('[VERSION]', f'[{tag}]')
        out = src.replace(original_block, blk, 1)
        out = out.replace('import argparse', 'IOU_THRESH = 0.5\n\nimport argparse', 1)
        open(fn, 'w').write(out)
        print(f'SUCCESS: {fn}   (ranking = {rank})')

    !diff -u demo.py     demo_nms.py    > demo_nms.patch
    !diff -u demo.py     demo_nms_v2.py > demo_nms_v2.patch
    !diff -u demo_nms.py demo_nms_v2.py > v1_to_v2.patch

    print('\n=== improvement 1 -> improvement 2 ===')
    print(open('v1_to_v2.patch').read())

SUCCESS: demo_nms.py   (ranking = lambda c: c['conf'])
SUCCESS: demo_nms_v2.py   (ranking = lambda c: (c['n_valid'], c['conf']))

=== improvement 1 -> improvement 2 ===
--- demo_nms.py	2026-07-29 12:46:58.459391768 +0000
+++ demo_nms_v2.py	2026-07-29 12:46:58.459391768 +0000
@@ -124,12 +124,12 @@
                 return inter / union if union > 0 else 0.0
 
             filtered = []
-            for c in sorted(candidates, key=lambda c: c['conf'], reverse=True):
+            for c in sorted(candidates, key=lambda c: (c['n_valid'], c['conf']), reverse=True):
                 if all(_iou(c['bbox'], k['bbox']) < IOU_THRESH for k in filtered):
                     filtered.append(c)
 
             if len(candidates) > len(filtered):
-                print(f'[NMS] {os.path.basename(str(img_path))} [v1-conf]: '
+                print(f'[NMS] {os.path.basename(str(img_path))} [v2-count]: '
                       f'{len(candidates)} -> {len(filtered)} | '
                       f'kept is_righ

In [ ]:
!python -u demo_nms.py    --img_folder my_data --out_folder out_nms_v1 \
    --side_view --save_mesh --full_frame 2>&1 | tee nms_v1_log.txt

!python -u demo_nms_v2.py --img_folder my_data --out_folder out_nms_v2 \
    --side_view --save_mesh --full_frame 2>&1 | tee nms_v2_log.txt

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
apex is not installed
apex is not installed
apex is not installed
/usr/local/lib/python3.12/dist-packages/mmcv/cnn/bricks/transformer.py:27: UserWarning: Fail to import ``MultiScaleDeformableAttention`` from ``mmcv.ops.multi_scale_deform_attn``, You should install ``mmcv-full`` if you need this module. 
  warnings.warn('Fail to import ``MultiScaleDeformableAttention`` from '
Use load_from_local loader
The model and loaded state dict do not match exactly

unexpected key in source state_dict: backbone.blocks.0.mlp.experts.0.weight, backbone.blocks.0.mlp.experts.0.bias, backbone.blocks.0.mlp.experts.1.weight, backbone.blocks.0.mlp.experts.1.bias, backbone.blocks.0.mlp.experts.2.weight, backbone.blocks.0.mlp.experts.2.b

In [ ]:
import glob, os, re, shutil, pandas as pd
from pathlib import Path
from datetime import datetime
from google.colab import files

stems = sorted(p.stem for p in Path('my_data').iterdir()
               if p.suffix.lower() in ('.jpg', '.jpeg', '.png'))
tbl = {}

# mesh count (precisely count 0)
for label, folder in [('baseline', 'out_debug'),
                      ('nms_v1',   'out_nms_v1'),
                      ('nms_v2',   'out_nms_v2')]:
    if not os.path.exists(folder):
        continue
    c = dict.fromkeys(stems, 0)
    for p in glob.glob(f'{folder}/*.obj'):
        s = os.path.basename(p).rsplit('_', 1)[0]
        if s in c:
            c[s] += 1
    tbl[label] = c

# final selected hand-side
for label, log in [('side_v1', 'nms_v1_log.txt'), ('side_v2', 'nms_v2_log.txt')]:
    if not os.path.exists(log):
        continue
    d = dict.fromkeys(stems, '-')
    for line in open(log):
        if m := re.search(r'\[NMS\] (\S+?)\.\w+ \[.*kept is_right=\[(\d+)\]', line):
            if m.group(1) in d:
                d[m.group(1)] = 'R' if m.group(2) == '1' else 'L'
    tbl[label] = d

df = pd.DataFrame(tbl).reindex(stems)
df.index.name = 'image'
df.to_csv('results_mesh_counts.csv')
display(df)

# --- Backup ---
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
dst = f"/content/drive/MyDrive/hamer_final/run_{stamp}"
os.makedirs(dst, exist_ok=True)

for d in ['out_official', 'out_debug', 'out_nms_v1', 'out_nms_v2', 'my_data']:
    if os.path.exists(d):
        shutil.copytree(d, os.path.join(dst, d), dirs_exist_ok=True)
        print(d, ':', len(os.listdir(d)), 'files')

for f in ['debug_log.txt', 'nms_v1_log.txt', 'nms_v2_log.txt',
          'results_diagnosis.csv', 'results_mesh_counts.csv',
          'demo_debug.py', 'demo_nms.py', 'demo_nms_v2.py',
          'demo_nms.patch', 'demo_nms_v2.patch', 'v1_to_v2.patch']:
    if os.path.exists(f):
        shutil.copy(f, dst)

print('\nBackUPDone:', dst)

zip_path = shutil.make_archive(f'/content/hamer_final_{stamp}', 'zip', root_dir=dst)
print('zip Size:', os.path.getsize(zip_path) // 1024 // 1024, 'MB')
files.download(zip_path)

,baseline,nms_v1,nms_v2,side_v1,side_v2
image,,,,,
ambiguous_hand_1,0,0,0,-,-
fist_closeup_1,0,0,0,-,-
fist_closeup_2,0,0,0,-,-
normal_hand_1,2,1,1,L,L
normal_hand_2,2,1,1,L,L
palm_closeup_1,2,1,1,R,L
two_hands_overlap,0,0,0,-,-


out_official : 29 files
out_debug : 15 files
out_nms_v1 : 9 files
out_nms_v2 : 9 files
my_data : 7 files

BackUPDone: /content/drive/MyDrive/hamer_final/run_20260729_125157
zip Size: 32 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>